In [1]:
# Install dependencies if not installed
!pip install datasets transformers torch accelerate openai nltk tqdm

In [2]:
import torch
import time
import openai
import nltk
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm
from scipy.spatial.distance import cosine
from PIL import Image, ExifTags

/afs/csail.mit.edu/u/y/yuexing/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/afs/csail.mit.edu/u/y/yuexing/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [3]:
# prompt: use OpenAI GPT4o to extract first columns' paper title. My prompt: Please extract paper title from this sentence.

!pip uninstall -y openai
!pip install --upgrade openai

import openai
print(openai.__version__)
import pandas as pd

# Assuming 'senior_author' DataFrame is loaded and contains a column named 'Paper Title'

# Set your OpenAI API key
openai.# Replace with your actual API key

Found existing installation: openai 1.65.2
Uninstalling openai-1.65.2:
  Successfully uninstalled openai-1.65.2
     |████████████████████████████████| 473 kB 7.6 MB/s eta 0:00:01
1.65.2


## Head-QA

In [3]:
from datasets import load_dataset

# Load dataset with trust enabled
dataset = load_dataset("dvilares/head_qa", split="train", trust_remote_code=True)

# Check dataset structure
print("Available columns:", dataset.column_names)
print("Dataset length:", len(dataset))

Available columns: ['name', 'year', 'category', 'qid', 'qtext', 'ra', 'image', 'answers']
Dataset length: 2657


In [5]:
# Load dataset with trust enabled
dataset = load_dataset("dvilares/head_qa",'en',  split="train", trust_remote_code=True)

# Check dataset structure
print("Available columns:", dataset.column_names)

# Extract relevant columns
dataset = dataset.select_columns(["qtext", "answers", "ra"]).select(range(500))  # Selecting only question and answer


Available columns: ['name', 'year', 'category', 'qid', 'qtext', 'ra', 'image', 'answers']


## Direct Prediction

In [6]:
import openai
import time

# Function: Direct Prediction with Confidence Level + Token Usage
def generate_direct_prediction(question, answers):
    """Predict the correct answer directly, provide a confidence score, and track token usage."""

    # Format answer choices
    answer_options = "\n".join([f"{ans['aid']}: {ans['atext']}" for ans in answers])

    # GPT-4o Prompt
    prompt = f"""
    Medical Question: "{question}"
    
    Available Answer Options:
    {answer_options}

    Your task:
    - Select the best answer (aid) directly.
    - Even if uncertain, choose the most probable option.
    - Always provide a confidence level (0-1), even if it is low.

    Response Format (strictly follow this format):
    - Predicted Answer: [aid] - [atext] (Confidence: X)

    Important:
    - Do NOT leave the answer blank.
    - If two options seem equally good, select the one that aligns best with common medical knowledge.
"""

    # Initialize OpenAI client
    client = openai.OpenAI(#Set the API key. See the how-to guide for further instructions
    )  # Set your API key

    # Measure start time for inference
    start_time = time.time()
    
    # Generate response using GPT-4o
    response = client.chat.completions.create(
        model="o1",
        messages=[{"role": "user", "content": prompt}],
#         temperature=0.5,
#         max_tokens=500
    )

    # Measure end time
    end_time = time.time()

    # Extract response text
    reasoning = response.choices[0].message.content

    # Extract token usage details
    input_tokens = response.usage.prompt_tokens  # Tokens in user input
    output_tokens = response.usage.completion_tokens  # Tokens in final output
    reasoning_tokens = max(0, output_tokens - int(output_tokens * 0.2))  # Estimate reasoning tokens

    # Extract prediction and confidence score
    predicted_aid, predicted_atext, confidence_score = None, None, None
    response_lines = reasoning.split("\n")

    for line in response_lines:
        if "Predicted Answer:" in line:
            parts = line.split(":")
            if len(parts) > 1:
                predicted_option = parts[1].strip().split(" - ")
                if len(predicted_option) == 2:
                    predicted_aid, predicted_atext = predicted_option
        elif "(Confidence:" in line:
            try:
                confidence_score = float(line.split("(Confidence:")[1].replace(")", "").strip())
            except ValueError:
                confidence_score = None  # Handle errors in parsing confidence

    return (
        predicted_aid, 
        predicted_atext, 
        confidence_score, 
        end_time - start_time, 
        input_tokens, 
        reasoning_tokens, 
        output_tokens
    )


In [7]:
import csv
import pandas as pd
from collections import Counter
from tqdm import tqdm
import os

# Define the CSV file name
csv_filename = "qtext_majority_predictions.csv"

# Define the column names (adding token tracking)
fieldnames = [
    "qtext", "correct_aid", "majority_predicted_aid",
    "majority_votes", "average_confidence", "average_inference_time",
    "average_input_tokens", "average_reasoning_tokens", "average_output_tokens"
]

# Ensure the CSV file exists and write the header if needed
if not os.path.exists(csv_filename):
    with open(csv_filename, mode="w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()  # Write header only once

# Run experiment with majority voting
with open(csv_filename, mode="a", newline="", encoding="utf-8", buffering=1) as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)

    for sample in tqdm(dataset):  # Iterate over the dataset
        question = sample["qtext"]
        answer_options = sample["answers"]  # List of answer choices (aid & atext)
        correct_answer = sample["ra"]  # Ground truth answer

        # Store predictions and metadata over multiple runs
        predictions = []
        confidence_scores = []
        inference_times = []
        input_tokens_list = []
        reasoning_tokens_list = []
        output_tokens_list = []

        for _ in range(5):  # Run the model 10 times per question
            direct_aid, direct_atext, direct_confidence, direct_time, in_tokens, reason_tokens, out_tokens = \
                generate_direct_prediction(question, answer_options)

            if direct_aid:  # Ensure a valid prediction was made
                predictions.append(direct_aid)
                confidence_scores.append(direct_confidence)
                inference_times.append(direct_time)
                input_tokens_list.append(in_tokens)
                reasoning_tokens_list.append(reason_tokens)
                output_tokens_list.append(out_tokens)

        # Determine the majority vote
        if predictions:
            majority_aid, majority_votes = Counter(predictions).most_common(1)[0]  # Most frequent prediction

            # Compute average confidence
            valid_confidences = [c for c in confidence_scores if c is not None]
            avg_confidence = sum(valid_confidences) / len(valid_confidences) if valid_confidences else None

            # Compute average inference time
            valid_inference_times = [t for t in inference_times if t is not None]
            avg_inference_time = sum(valid_inference_times) / len(valid_inference_times) if valid_inference_times else None

            # Compute average token usage
            avg_input_tokens = sum(input_tokens_list) / len(input_tokens_list) if input_tokens_list else None
            avg_reasoning_tokens = sum(reasoning_tokens_list) / len(reasoning_tokens_list) if reasoning_tokens_list else None
            avg_output_tokens = sum(output_tokens_list) / len(output_tokens_list) if output_tokens_list else None
        else:
            majority_aid, majority_votes = None, 0
            avg_confidence, avg_inference_time = None, None
            avg_input_tokens, avg_reasoning_tokens, avg_output_tokens = None, None, None

        # Store the result row
        result_row = {
            "qtext": question,
            "correct_aid": correct_answer,
            "majority_predicted_aid": majority_aid,
            "majority_votes": majority_votes,
            "average_confidence": avg_confidence,
            "average_inference_time": avg_inference_time,
            "average_input_tokens": avg_input_tokens,
            "average_reasoning_tokens": avg_reasoning_tokens,
            "average_output_tokens": avg_output_tokens
        }

        # Append the result row to the CSV file in real-time
        writer.writerow(result_row)
        file.flush()  # Ensure immediate write to file

# Display a preview of the saved results
results_df = pd.read_csv(csv_filename)
print(results_df.head(10))


 31%|███████████▎                         | 153/500 [1:20:08<3:01:45, 31.43s/it]


KeyboardInterrupt: 

In [ ]:
import pandas as pd

# Load the results CSV
direct_df = pd.read_csv("qtext_majority_predictions.csv")

# Ensure string format and strip whitespace
direct_df["majority_predicted_aid"] = direct_df["majority_predicted_aid"].astype(str).str.strip()
direct_df["correct_aid"] = direct_df["correct_aid"].astype(str).str.strip()

# Handle missing values before comparison
direct_df = direct_df.dropna(subset=["majority_predicted_aid", "correct_aid"])  # Remove rows with NaN values

# Compute accuracy only on valid (non-null) predictions
correct_predictions = (direct_df["majority_predicted_aid"] == direct_df["correct_aid"]).sum()
total_predictions = len(direct_df)

accuracy = (correct_predictions / total_predictions) * 100 if total_predictions > 0 else 0  # Avoid division by zero

# Compute mean and standard deviation for tokens & inference time
metrics = ["average_input_tokens", "average_reasoning_tokens", "average_output_tokens", "average_inference_time"]
stats = {}

for metric in metrics:
    if metric in direct_df:
        stats[metric] = {
            "mean": direct_df[metric].dropna().mean(),
            "std": direct_df[metric].dropna().std()
        }
    else:
        stats[metric] = {"mean": None, "std": None}

# Display results
print("\n===== Experiment Summary =====")
print(f"Total Predictions: {total_predictions}")
print(f"Correct Predictions: {correct_predictions}")
print(f"Accuracy: {accuracy:.2f}%")

for metric, values in stats.items():
    mean_val, std_val = values["mean"], values["std"]
    metric_name = metric.replace("average_", "").replace("_", " ").title()  # Formatting for readability
    if mean_val is not None:
        print(f"{metric_name}: {mean_val:.2f} ± {std_val:.2f}")
    else:
        print(f"{metric_name}: No data available")

# print("\n===== Last 10 Predictions =====")
# print(direct_df.tail(10))


## With reasoning

In [14]:
import openai
import time

# Function to generate prediction (aid & atext), reasoning, and track token usage
def generate_prediction_with_reasoning(question, answers):
    """Predict the correct answer and generate reasoning while tracking token usage."""
    
    # Format answer options into a string
    answer_options = "\n".join([f"{ans['aid']}: {ans['atext']}" for ans in answers])

    prompt = f"""
    Medical Question: "{question}"
    
    Available Answer Options (Select only from these):
    {answer_options}
    
    Task:
    - Select the most appropriate answer (aid) based on the provided options.
    - Justify your choice in a step-wise reasoning process.
    - Assign a confidence level (0-100%) for each step, indicating certainty at that stage.

    Response Format (Strictly follow this format):
    - Predicted Correct Answer: [aid] - [atext]
    - Step 1: [reasoning] (Confidence: X%)
    - Step 2: [reasoning] (Confidence: Y%)
    - ...
    - Final Decision: [aid] - [atext] (Confidence: Z%)

    Important:
    - You must select an answer from the available options.
    - Do NOT leave confidence levels blank.
    - Ensure the reasoning follows a logical progression towards the final decision.
"""

    # Initialize OpenAI client
    client = openai.OpenAI(#Set the API key. See the how-to guide for further instructions
)  # Generate response using GPT-4o
    # Measure start time for inference
    start_time = time.time()
    
    # Generate response using GPT-4o
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],   
        temperature=0.5,
        max_tokens=500 
    )

    # Measure end time
    end_time = time.time()
    
    # Extract response text
    reasoning = response.choices[0].message.content

    # Extract token usage details
    input_tokens = response.usage.prompt_tokens  # Tokens in user input
    output_tokens = response.usage.completion_tokens  # Tokens in final output
    reasoning_tokens = max(0, output_tokens - int(output_tokens * 0.2))  # Approximate reasoning tokens

    # Extract prediction and reasoning
    response_lines = reasoning.split("\n")
    predicted_aid = ""
    predicted_atext = ""
    reasoning_process = []

    for line in response_lines:
        if "Predicted Correct Answer:" in line or "Final Decision:" in line:
            parts = line.split(":")
            if len(parts) > 1:
                predicted_option = parts[1].strip().split(" - ")
                if len(predicted_option) == 2:
                    predicted_aid, predicted_atext = predicted_option
        elif "Step" in line:
            reasoning_process.append(line.strip())

    # Join reasoning steps
    reasoning_text = " | ".join(reasoning_process)
    
    return (
        predicted_aid, 
        predicted_atext, 
        reasoning_text, 
        end_time - start_time, 
        input_tokens, 
        reasoning_tokens, 
        output_tokens
    )


In [15]:
import csv
import pandas as pd
import os
from tqdm import tqdm

# Define CSV file path
csv_file = "qtext_to_aid_predictions.csv"

# Ensure the file has headers if it doesn't exist
if not os.path.exists(csv_file):
    with open(csv_file, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "qtext", "predicted_correct_aid", "ra", "reasoning_process", "inference_time",
            "input_tokens", "reasoning_tokens", "output_tokens"
        ])  # Write header once

# Run experiment and append results in real-time
with open(csv_file, mode="a", newline="", encoding="utf-8", buffering=1) as f:
    writer = csv.DictWriter(f, fieldnames=[
        "qtext", "predicted_correct_aid", "ra", "reasoning_process", "inference_time",
        "input_tokens", "reasoning_tokens", "output_tokens"
    ])
    
    for sample in tqdm(dataset):  # Iterate through dataset
        question = sample["qtext"]
        answer_options = sample["answers"]  # List of answer choices (each has aid & atext)
        correct_answer = sample["ra"]  # Ground-truth correct answer (atext)

        # Generate prediction, reasoning, inference time, and token usage
        predicted_aid, predicted_atext, reasoning_process, inference_time, input_tokens, reasoning_tokens, output_tokens = \
            generate_prediction_with_reasoning(question, answer_options)

        # Store results in real-time
        writer.writerow({
            "qtext": question,
            "predicted_correct_aid": predicted_aid,
            "ra": correct_answer,
            "reasoning_process": reasoning_process,
            "inference_time": inference_time,
            "input_tokens": input_tokens,
            "reasoning_tokens": reasoning_tokens,
            "output_tokens": output_tokens
        })
        f.flush()  # Force immediate write to CSV

# Load and display the last 10 results
direct_df = pd.read_csv(csv_file)
print(direct_df.tail(10))  # Display the last 10 rows


100%|█████████████████████████████████████████| 100/100 [07:39<00:00,  4.60s/it]

                                                qtext  predicted_correct_aid  \
90  What enzyme does a retrovirus use to synthesiz...                      5   
91  How many palindromic sequences exist on the hu...                      1   
92                                  It is an oncogen:                      4   
93                It is a stop codon for translation:                      3   
94  In the replication of eukaryotic DNA the funct...                      2   
95  If the guanine percentage of a double-stranded...                      5   
96  Among the 5 types of histones of chromatin, th...                      4   
97          The three-dimensional structure of B-DNA:                      3   
98  Williams-Beuren syndrome is characterized by a...                      5   
99  The genetic phenomenon that explains that the ...                      4   

    ra                                  reasoning_process  inference_time  \
90   5  - Step 1: Understand the function 

In [17]:
import pandas as pd

# Ensure string format and strip whitespace
direct_df["predicted_correct_aid"] = direct_df["predicted_correct_aid"].astype(str).str.strip()
direct_df["ra"] = direct_df["ra"].astype(str).str.strip()

# Handle missing values before comparison
direct_df = direct_df.dropna(subset=["predicted_correct_aid", "ra"])  # Remove rows with NaN values

# Compute accuracy only on valid (non-null) predictions
correct_predictions = (direct_df["predicted_correct_aid"] == direct_df["ra"]).sum()
total_predictions = len(direct_df)

accuracy = (correct_predictions / total_predictions) * 100 if total_predictions > 0 else 0  # Avoid division by zero

# Display results
print(f"Total Predictions: {total_predictions}")
print(f"Correct Predictions: {correct_predictions}")
print(f"Accuracy: {accuracy:.2f}%")

Total Predictions: 100
Correct Predictions: 93
Accuracy: 93.00%


In [18]:
# Compute average tokens and inference time while ignoring NaN values
avg_input_tokens = direct_df["input_tokens"].dropna().mean() if "input_tokens" in direct_df else None
avg_reasoning_tokens = direct_df["reasoning_tokens"].dropna().mean() if "reasoning_tokens" in direct_df else None
avg_output_tokens = direct_df["output_tokens"].dropna().mean() if "output_tokens" in direct_df else None
avg_inference_time = direct_df["inference_time"].dropna().mean() if "inference_time" in direct_df else None

# Compute standard deviations for better analysis
std_input_tokens = direct_df["input_tokens"].std() if "input_tokens" in direct_df else None
std_reasoning_tokens = direct_df["reasoning_tokens"].std() if "reasoning_tokens" in direct_df else None
std_output_tokens = direct_df["output_tokens"].std() if "output_tokens" in direct_df else None
std_inference_time = direct_df["inference_time"].std() if "inference_time" in direct_df else None

# Display results
print("\n===== Experiment Summary =====")
print(f"Total Predictions: {len(direct_df)}")
print(f"Average Input Tokens: {avg_input_tokens:.2f} ± {std_input_tokens:.2f}" if avg_input_tokens is not None else "No input token data")
print(f"Average Reasoning Tokens: {avg_reasoning_tokens:.2f} ± {std_reasoning_tokens:.2f}" if avg_reasoning_tokens is not None else "No reasoning token data")
print(f"Average Output Tokens: {avg_output_tokens:.2f} ± {std_output_tokens:.2f}" if avg_output_tokens is not None else "No output token data")
print(f"Average Inference Time: {avg_inference_time:.2f} sec ± {std_inference_time:.2f}" if avg_inference_time is not None else "No inference time data")



===== Experiment Summary =====
Total Predictions: 100
Average Input Tokens: 241.23 ± 10.51
Average Reasoning Tokens: 162.31 ± 33.04
Average Output Tokens: 202.33 ± 41.32
Average Inference Time: 4.57 sec ± 1.23


## Previous Testings


In [20]:
def generate_stepwise_reasoning(question):
    """Generate reasoning steps, predict answer, and assign confidence scores."""
    prompt = f"""
    Given the following medical question: "{question}"
    Provide a step-wise reasoning process to determine the correct answer.
    For each step, rate your confidence level on a scale of 0 to 1.

    Format:
    Step 1: [reasoning] (Confidence: X)
    Step 2: [reasoning] (Confidence: Y)
    ...
    Final Answer: [Predicted Answer] (Confidence: Z)
    """
    
    client = openai.OpenAI(
    #Set the API key. See the how-to guide for further instructions
)  # Generate response using GPT-4o
    response = client.chat.completions.create(
        model="gpt-4",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        max_tokens=500
    )
    
    # Extract response text
    reasoning = response.choices[0].message.content

    return reasoning.strip()

In [21]:
# Run experiment
results = []

for sample in tqdm(dataset.select(range(50))):  # Run on 50 samples for quick testing
    question = sample["qtext"]
    correct_answer = sample["atext"]  # Ground-truth answer

    # Measure inference time
    start_time = time.time()
    reasoning_output = generate_stepwise_reasoning(question)
    end_time = time.time()

    # Store results
    results.append({
        "question": question,
        "predicted_reasoning": reasoning_output,
        "correct_answer": correct_answer,
        "inference_time": end_time - start_time
    })

# Convert results to DataFrame and save
results_df = pd.DataFrame(results)
results_df.to_csv("stepwise_reasoning_results.csv", index=False)

# Display results
print(results_df.head(10))

  0%|                                                    | 0/50 [00:00<?, ?it/s]


KeyError: 'atext'

In [ ]:

# Run experiment for different reasoning depths
results = []

for depth in REASONING_DEPTHS:
    accuracy_list = []
    inference_times = []
    redundancy_scores = []
    hallucination_counts = 0

    for sample in tqdm(dataset.select(range(50))):  # Use first 50 samples for quick testing
        question = sample["qtext"]
        correct_answer = sample["ra"]

        # Measure inference time
        start_time = time.time()
        response = generate_response(question, depth=depth)
        end_time = time.time()

        # Compute accuracy (binary match)
        accuracy = 1 if correct_answer.lower() in response.lower() else 0

        # Compute redundancy score
        steps = response.split("\n")[:depth]  # Extract stepwise reasoning
        redundancy_score = compute_redundancy_score(steps)

        # Check for hallucinations (manually defined)
        hallucinations = sum(1 for step in steps if "incorrect" in step.lower() or "not real" in step.lower())

        # Store results
        accuracy_list.append(accuracy)
        inference_times.append(end_time - start_time)
        redundancy_scores.append(redundancy_score)
        hallucination_counts += hallucinations

    # Store aggregate results
    results.append({
        "depth": depth,
        "accuracy": np.mean(accuracy_list),
        "inference_time": np.mean(inference_times),
        "redundancy_score": np.mean(redundancy_scores),
        "hallucination_rate": hallucination_counts / len(dataset.select(range(50)))
    })

# Convert results to DataFrame and save as CSV
results_df = pd.DataFrame(results)
results_df.to_csv("reasoning_experiment_results.csv", index=False)

# Display results
print(results_df)

## PubMedQA

In [2]:
from datasets import load_dataset

# Load the PubMedQA dataset
dataset = load_dataset("bigbio/pubmed_qa")

# Display dataset details
print(dataset)

/afs/csail.mit.edu/u/y/yuexing/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/afs/csail.mit.edu/u/y/yuexing/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


README.md:   0%|          | 0.00/2.36k [00:00<?, ?B/s]

pubmed_qa.py:   0%|          | 0.00/10.3k [00:00<?, ?B/s]

bigbiohub.py:   0%|          | 0.00/19.3k [00:00<?, ?B/s]

The repository for bigbio/pubmed_qa contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/bigbio/pubmed_qa.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


pqaa.zip:   0%|          | 0.00/156M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['QUESTION', 'CONTEXTS', 'LABELS', 'MESHES', 'YEAR', 'reasoning_required_pred', 'reasoning_free_pred', 'final_decision', 'LONG_ANSWER'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['QUESTION', 'CONTEXTS', 'LABELS', 'MESHES', 'YEAR', 'reasoning_required_pred', 'reasoning_free_pred', 'final_decision', 'LONG_ANSWER'],
        num_rows: 11269
    })
})


In [5]:
dataset_labeled = load_dataset("bigbio/pubmed_qa", name="pubmed_qa_labeled_fold1_source")

# View first sample
print(dataset_labeled["train"][0])

pqal.zip:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

{'QUESTION': "Is cytokeratin immunoreactivity useful in the diagnosis of short-segment Barrett's oesophagus in Korea?", 'CONTEXTS': ["Cytokeratin 7/20 staining has been reported to be helpful in diagnosing Barrett's oesophagus and gastric intestinal metaplasia. However, this is still a matter of some controversy.", "To determine the diagnostic usefulness of cytokeratin 7/20 immunostaining for short-segment Barrett's oesophagus in Korea.", "In patients with Barrett's oesophagus, diagnosed endoscopically, at least two biopsy specimens were taken from just below the squamocolumnar junction. If goblet cells were found histologically with alcian blue staining, cytokeratin 7/20 immunohistochemical stains were performed. Intestinal metaplasia at the cardia was diagnosed whenever biopsy specimens taken from within 2 cm below the oesophagogastric junction revealed intestinal metaplasia. Barrett's cytokeratin 7/20 pattern was defined as cytokeratin 20 positivity in only the superficial gland, co

In [7]:
for example in dataset_labeled["train"].select(range(3)):
    print("Q:", example["QUESTION"])  # Use 'QUESTION' instead of 'question'
    print("Context:", " ".join(example["CONTEXTS"])[:200])  # Join context sentences
    print("Answer:", example["LONG_ANSWER"])  # Use 'LONG_ANSWER' instead of 'long_answer'
    print("Label:", example["final_decision"])  # Use 'final_decision' instead of 'label'
    print("-" * 80)


Q: Is cytokeratin immunoreactivity useful in the diagnosis of short-segment Barrett's oesophagus in Korea?
Context: Cytokeratin 7/20 staining has been reported to be helpful in diagnosing Barrett's oesophagus and gastric intestinal metaplasia. However, this is still a matter of some controversy. To determine the di
Answer: Barrett's cytokeratin 7/20 pattern can be a useful marker for the diagnosis of short-segment Barrett's oesophagus, although the false positive or false negative rate is approximately 25%.
Label: yes
--------------------------------------------------------------------------------
Q: Is extended aortic replacement in acute type A dissection justifiable?
Context: The aim of this study was to evaluate the effectiveness of our surgical strategy for acute aortic dissection based on the extent of the dissection and the site of the entry, with special emphasis on r
Answer: Extended replacement of the dissected ascending aorta and aortic arch can be done with good early and m

## Reasoning from Question to Final Decision using ChatGPT 4o

In [9]:
from openai import OpenAI  # This will cause an ImportError
# import openai

#We define which model to use throughout
MODEL = 'gpt-4o'
MAX_TOKENS = 8000
WAIT_TIME = 0.8 # Wait time between each request. This depends on the rate limit of the model used: GPT-4 needs longer wait time than GPT-3.5.

client = openai.OpenAI(
    #Set the API key. See the how-to guide for further instructions
)

In [13]:
for example in dataset_labeled["train"].select(range(3)):
    question = example["QUESTION"]
    contexts = " ".join(example["CONTEXTS"])  # Combine context sentences
    final_decision = example["final_decision"]  # Yes, No, or Maybe

    # Define the GPT-4o prompt
    prompt = f"""
    Based on this question: "{question}"
    
    I want to know whether the final decision is Yes, No, or Maybe. Please provide a tree-of-thought explanation following NCCN's guideline.

    Context:
    {contexts}

    Please provide the specific NCCN guideline that you are following and its link.
    """

    client = openai.OpenAI(
    #Set the API key. See the how-to guide for further instructions
)  # Generate response using GPT-4o
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "system", "content": "You are a medical AI assistant specializing in evidence-based oncology decision-making."},
                  {"role": "user", "content": prompt}]
    )

    # Extract response text
    reasoning = response.choices[0].message.content

    # Display output
    print(f"Question: {question}")
    print(f"Final Decision: {final_decision}")
    print("Tree-of-Thought Explanation (GPT-4o):")
    print(reasoning)
    print("-" * 80)


Question: Is cytokeratin immunoreactivity useful in the diagnosis of short-segment Barrett's oesophagus in Korea?
Final Decision: yes
Tree-of-Thought Explanation (GPT-4o):
To determine whether cytokeratin immunoreactivity is useful in the diagnosis of short-segment Barrett's oesophagus in Korea, we need to follow a structured decision-making process based on evidence and guidelines such as those provided by the National Comprehensive Cancer Network (NCCN). Unfortunately, specific sections of NCCN guidelines on gastrointestinal cancers, including those focused specifically on Barrett's esophagus, are proprietary and require access through institutional subscriptions or professional memberships. As such, I cannot provide a direct link or full guideline here, but I can guide you conceptually through an evidence-based thought process.

Here's a tree-of-thought explanation:

1. **Understanding the Evidence:**
   - **Sensitivity and Specificity:** The cytokeratin 7/20 staining pattern shows 